In [1]:
!pip install -r requirements.txt

In [2]:
import pandas as pd
import numpy as np
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.preprocessing import OrdinalEncoder

train_df = pd.read_csv("train-test.csv")
val_df = pd.read_csv("validation.csv")
dec_df = pd.read_csv("december-chart-inputs.csv")

train_df['weight'] = train_df['weight'].abs()
train_df['weight'] = train_df['weight'].fillna(train_df['weight'].median())

val_df['weight'] = val_df['weight'].fillna(train_df['weight'].median())
dec_df['weight'] = dec_df['weight'].fillna(train_df['weight'].median())

for df in [train_df, val_df, dec_df]:
    df['date'] = pd.to_datetime(df['date'])
    df['month'] = df['date'].dt.month
    df['day_of_week'] = df['date'].dt.dayofweek
    df['day_of_year'] = df['date'].dt.dayofyear

cat_cols = ['pickup', 'delivery', 'equipment']
encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)

train_df[cat_cols] = encoder.fit_transform(train_df[cat_cols].astype(str))
val_df[cat_cols] = encoder.transform(val_df[cat_cols].astype(str))
dec_df[cat_cols] = encoder.transform(dec_df[cat_cols].astype(str))

features = ['pickup', 'delivery', 'distance', 'equipment', 'weight', 'month', 'day_of_week', 'day_of_year']
X_train = train_df[features]
y_train = train_df['posted_rate']

model = HistGradientBoostingRegressor(random_state=42)
model.fit(X_train, y_train)

X_val = val_df[features]
val_preds = model.predict(X_val)

X_dec = dec_df[features]
dec_preds = model.predict(X_dec)

template = pd.read_csv("validation-predictions-template.csv")
template['predicted_rate'] = val_preds
template.to_csv("validation_predictions.csv", index=False)

dec_df_out = pd.read_csv("december-chart-inputs.csv")
dec_df_out['predicted_rate'] = dec_preds
dec_df_out.to_csv("december-chart-inputs.csv", index=False)

In [3]:
!python score.py --predictions validation_predictions.csv --december-predictions december-chart-inputs.csv

Validated 12,000 final predictions.
Validated 31 fixed December predictions.
Created chart: scorer_results/candidate_december.png
Final validation metrics are calculated by Spotter after submission.
